# DanioDecima Training Results Analysis

### This notebook analyzes the results from DanioDecima model training experiments.

### Generate visualization of training and evaluation loss curves for a hyperparameter sweep over the learning rate. Figure 2 in the DanioDecima manuscript.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorboard.backend.event_processing import event_accumulator
import glob
import re
import colorsys

class ExperimentAnalyzer:
    def __init__(self):
        self.experiment_data = {}
        self.all_metrics = set()
    
    def load_multiple_dirs(self, base_dirs):
        """Load experiments from multiple base directories"""
        all_experiment_dirs = []
        for base_dir in base_dirs:
            dirs = self._find_experiment_dirs(base_dir)
            all_experiment_dirs.extend(dirs)
        
        print(f"Total experiment directories found: {len(all_experiment_dirs)}")
        self.experiment_data, self.all_metrics = self._load_experiment_data(all_experiment_dirs)
        print(f"Loaded {len(self.experiment_data)} experiments with metrics: {sorted(self.all_metrics)}")
        
    def _find_experiment_dirs(self, base_dir):
        """Find all experiment directories - fixed to match your original logic"""
        array_dirs = glob.glob(os.path.join(base_dir, "*"))
        experiment_dirs = []
        
        print(f"Scanning for experiments in {base_dir}...")
        
        for array_dir in array_dirs:
            if not os.path.isdir(array_dir):
                continue
                
            # Look for task directories
            task_dirs = glob.glob(os.path.join(array_dir, "task_*"))
            
            if task_dirs:  # Ray Tune structure
                for task_dir in task_dirs:
                    print(f"  Checking task directory: {task_dir}")
                    
                    # Find trial directories (hyperparameter combinations) - this was the key missing part!
                    trial_dirs = glob.glob(os.path.join(task_dir, "lr_*")) + \
                               glob.glob(os.path.join(task_dir, "train_*"))
                    
                    print(f"    Found {len(trial_dirs)} trial directories")
                    
                    for trial_dir in trial_dirs:
                        print(f"      Checking trial: {trial_dir}")
                        
                        # Look for version directories or directly for event files
                        version_dirs = glob.glob(os.path.join(trial_dir, "version_*"))
                        
                        if version_dirs:
                            for version_dir in version_dirs:
                                experiment_dirs.append(version_dir)
                                print(f"        Added version dir: {version_dir}")
                        else:
                            if glob.glob(os.path.join(trial_dir, "events.out.tfevents.*")):
                                experiment_dirs.append(trial_dir)
                                print(f"        Added trial dir: {trial_dir}")
            else:
                # Check if the array directory itself contains event files
                event_files = glob.glob(os.path.join(array_dir, "events.out.tfevents.*"))
                if event_files:
                    experiment_dirs.append(array_dir)
                    print(f"  Added array dir: {array_dir}")
        
        print(f"Found {len(experiment_dirs)} total experiment directories")
        return experiment_dirs
    
    def _calculate_early_stopping_metrics(self, df, patience=10, epsilon=0.001, minimize=True):
        """
        Calculate early stopping metrics properly using patience and epsilon
        
        Parameters:
        -----------
        df : DataFrame
            DataFrame with 'epoch' and 'value' columns
        patience : int
            Number of epochs to wait without improvement
        epsilon : float
            Minimum change to qualify as improvement
        minimize : bool
            Whether we're minimizing (loss) or maximizing (accuracy)
        """
        if df.empty:
            return None
        
        # Sort by epoch to ensure chronological order
        df_sorted = df.sort_values('epoch').reset_index(drop=True)
        
        best_value = None
        best_epoch = None
        best_step = None
        patience_counter = 0
        early_stop_epoch = None
        
        for idx, row in df_sorted.iterrows():
            current_value = row['value']
            current_epoch = row['epoch']
            current_step = row['step']
            
            # Check if this is an improvement
            is_improvement = False
            if best_value is None:
                is_improvement = True
            else:
                if minimize:
                    # For loss: improvement if current value is significantly lower
                    is_improvement = (best_value - current_value) > epsilon
                else:
                    # For accuracy/pearson: improvement if current value is significantly higher
                    is_improvement = (current_value - best_value) > epsilon
            
            if is_improvement:
                best_value = current_value
                best_epoch = current_epoch
                best_step = current_step
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Check if we should stop
            if patience_counter >= patience:
                early_stop_epoch = current_epoch
                break
        
        return {
            'best_value': best_value,
            'best_epoch': best_epoch,
            'best_step': best_step,
            'early_stop_epoch': early_stop_epoch,
            'patience_exhausted': patience_counter >= patience
        }

    def _extract_experiment_group(self, path):
        """Extract experiment group from directory path"""
        if 'decima_tune_19136570' in path or 'decima_tune_19040183' in path:
            return 'Random Init LR Sweep'
        elif 'decima_tune_19039194' in path:
            return 'Human-Borzoi LR Sweep'
        elif 'decima_tune_19136584' in path:
            return 'Human-Borzoi Best LR (3e-5)'
        else:
            # Extract the experiment ID as fallback
            import re
            match = re.search(r'decima_tune_(\d+)', path)
            if match:
                return f'Experiment {match.group(1)}'
            return 'Unknown'

    def _load_experiment_data(self, experiment_dirs):
        """Load experiment data with group information"""
        experiment_data = {}
        all_metrics = set()
        
        for exp_dir in experiment_dirs:
            event_files = glob.glob(os.path.join(exp_dir, "events.out.tfevents.*"))
            
            if not event_files:
                continue
            
            params = self._extract_params_from_path(exp_dir)
            
            # Add experiment group
            exp_group = self._extract_experiment_group(exp_dir)
            params['experiment_group'] = exp_group
            
            # Create descriptive name
            if 'lr' in params and 'bs' in params:
                if 'weight' in params:
                    desc_name = f"lr={params['lr']:.2e}_bs={params['bs']}_w={params['weight']:.2e}"
                    if 'replicate' in params:
                        desc_name += f"_rep={params['replicate']}"
                else:
                    desc_name = f"lr={params['lr']:.2e}_bs={params['bs']}"
                    if 'replicate' in params:
                        desc_name += f"_rep={params['replicate']}"
            else:
                desc_name = os.path.basename(exp_dir)
                if 'replicate' in params:
                    desc_name += f"_rep={params['replicate']}"
            
            latest_event = sorted(event_files, key=os.path.getmtime)[-1]
            try:
                metrics_data, metrics = self._load_tb_file(latest_event)
                
                # Apply early stopping analysis (keeping existing code)
                early_stopping_patience = 10
                early_stopping_epsilon = 0.001
                
                for metric in metrics:
                    if metric in metrics_data:
                        df = metrics_data[metric]
                        if df.empty:
                            continue
                        
                        if 'loss' in metric.lower():
                            es_result = self._calculate_early_stopping_metrics(
                                df, patience=early_stopping_patience, 
                                epsilon=early_stopping_epsilon, minimize=True
                            )
                            
                            if es_result:
                                metrics_data[f"{metric}_early_stop"] = es_result
                                metrics_data[f"{metric}_min_value"] = es_result['best_value']
                                metrics_data[f"{metric}_min_step"] = es_result['best_step']
                                metrics_data[f"{metric}_min_epoch"] = es_result['best_epoch']
                
                all_metrics.update(metrics)
                
                experiment_data[desc_name] = {
                    'path': exp_dir,
                    'params': params,
                    'metrics': metrics_data
                }
                print(f"  Loaded {desc_name} [{exp_group}]")
            except Exception as e:
                print(f"  ERROR loading {latest_event}: {e}")
        
        return experiment_data, list(all_metrics)

    def plot_min_val_loss_vs_lr(self, figsize=(14, 8), use_early_stopped=True):
        """Plot minimum validation loss vs learning rate with experiment groups"""
        data_points = []
        
        for exp_name, data in self.experiment_data.items():
            if 'lr' not in data['params']:
                continue
                
            # Use early-stopped minimum if available
            if use_early_stopped and 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                min_loss = data['metrics']['val_loss']['value'].min()
            else:
                continue
                
            data_points.append({
                'experiment': exp_name,
                'lr': data['params']['lr'],
                'min_val_loss': min_loss,
                'weight': data['params'].get('weight', 'unknown'),
                'model_run': data['params'].get('replicate', 'unknown'),
                'experiment_group': data['params'].get('experiment_group', 'Unknown')
            })
        
        if not data_points:
            print("No data available for min val loss vs lr plot")
            return
        
        df = pd.DataFrame(data_points)
        
        plt.figure(figsize=figsize)
        
        # Define colors and markers for each group
        group_colors = {
            'Random Init LR Sweep': 'blue',
            'Human-Borzoi LR Sweep': 'red', 
            'Human-Borzoi Best LR (3e-5)': 'green'
        }
        
        group_markers = {
            'Random Init LR Sweep': 'o',
            'Human-Borzoi LR Sweep': 's',
            'Human-Borzoi Best LR (3e-5)': '^'
        }
        
        # Plot each group separately
        for group in df['experiment_group'].unique():
            group_df = df[df['experiment_group'] == group]
            
            # For the replicates group, use different markers for each model run
            if group == 'Human-Borzoi Best LR (3e-5)':
                sns.scatterplot(data=group_df, x='lr', y='min_val_loss', 
                            style='model_run', s=120, 
                            color=group_colors.get(group, 'gray'),
                            label=group)
            else:
                # For LR sweeps, use consistent markers
                plt.scatter(group_df['lr'], group_df['min_val_loss'], 
                        c=group_colors.get(group, 'gray'),
                        marker=group_markers.get(group, 'o'),
                        s=120, alpha=0.7, 
                        label=group, edgecolors='black', linewidth=0.5)
        
        plt.xscale('log')
        plt.xlabel('Learning Rate (log scale)', fontsize=12)
        plt.ylabel('Minimum Validation Loss (Early Stopped)', fontsize=12)
        plt.title('Early-Stopped Minimum Validation Loss vs Learning Rate\nGrouped by Experiment Type', fontsize=14)
        plt.grid(True, alpha=0.3)
        
        # Custom legend
        plt.legend(title='Experiment Groups', title_fontsize=12, fontsize=10, 
                bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Add text annotations for key insights
        plt.text(0.02, 0.98, 
                'Random Init: LR sweeps with random initialization\n'
                'Human-Borzoi LR Sweep: Single replicate LR sweep\n'
                'Human-Borzoi Best LR: 4 replicates at optimal LR', 
                transform=plt.gca().transAxes, 
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                fontsize=9)
        
        plt.tight_layout()
        return plt.gcf()
    
    def _extract_params_from_path(self, path):
        """Extract parameters including replicate information - fixed for task numbers"""
        params = {}
        
        # Extract learning rate - handle multiple formats
        lr_patterns = [r'lr[_=]([\d.e\-]+)', r'lr_([\d.e\-]+)']
        for pattern in lr_patterns:
            lr_match = re.search(pattern, path)
            if lr_match:
                params['lr'] = float(lr_match.group(1))
                break
        
        # Extract batch size
        bs_patterns = [r'bs[_=](\d+)', r'bs_(\d+)']
        for pattern in bs_patterns:
            bs_match = re.search(pattern, path)
            if bs_match:
                params['bs'] = int(bs_match.group(1))
                break
        
        # Extract weight parameter
        w_patterns = [r'w[_=]([\d.e\-]+)', r'w_([\d.e\-]+)']
        for pattern in w_patterns:
            w_match = re.search(pattern, path)
            if w_match:
                params['weight'] = float(w_match.group(1))
                break
        
        # Extract replicate information - prioritize task numbers for Ray Tune
        rep_patterns = [
            r'task_(\d+)',     # This should catch task_0, task_1, etc. - MOST IMPORTANT
            r'rep[_=](\d+)', 
            r'rep_(\d+)', 
            r'replicate[_=](\d+)',
            r'version_(\d+)',  # version directories as fallback
            r'trial_(\d+)',    # trial numbers
            r'run_(\d+)'       # run numbers
        ]
        for pattern in rep_patterns:
            rep_match = re.search(pattern, path)
            if rep_match:
                params['replicate'] = int(rep_match.group(1))
                break
        
        # Extract array job ID and task ID
        array_match = re.search(r'_(\d+)(?:_(\d+))?', path)
        if array_match:
            params['array_id'] = array_match.group(1)
            if array_match.group(2):
                params['task_id'] = array_match.group(2)
        
        return params

    def _load_tb_file(self, path):
        """Load tensorboard file with adaptive epoch handling"""
        ea = event_accumulator.EventAccumulator(path, size_guidance={'scalars': 0})
        ea.Reload()
        
        tags = ea.Tags()['scalars']
        dfs = {}
        
        # First pass: load all data and find epoch-based metrics
        epoch_based_metrics = []
        step_based_metrics = []
        actual_max_epoch = None
        
        for tag in tags:
            events = ea.Scalars(tag)
            dfs[tag] = pd.DataFrame([
                {'step': e.step, 'value': e.value, 'wall_time': e.wall_time}
                for e in events
            ])
            
            if len(dfs[tag]) > 0:
                max_step = dfs[tag]['step'].max()
                
                # Classify metrics by their characteristics
                if 'epoch' in tag.lower():
                    # Explicitly epoch-based metrics
                    epoch_based_metrics.append((tag, max_step))
                    if actual_max_epoch is None or max_step > actual_max_epoch:
                        actual_max_epoch = max_step
                elif max_step <= 50:  # Heuristic: small numbers are likely epochs
                    epoch_based_metrics.append((tag, max_step))
                    if actual_max_epoch is None or max_step > actual_max_epoch:
                        actual_max_epoch = max_step
                else:
                    step_based_metrics.append((tag, max_step))
        
        # If we found epoch-based metrics, use them to determine the actual max epoch
        if actual_max_epoch is None:
            # Fallback: estimate from the step-based metrics
            if step_based_metrics:
                # Assume reasonable steps per epoch (this is dataset dependent)
                # You might want to adjust this based on your specific setup
                estimated_steps_per_epoch = 1000  # Adjust this for your dataset
                max_steps = max(max_step for _, max_step in step_based_metrics)
                actual_max_epoch = max_steps / estimated_steps_per_epoch
            else:
                actual_max_epoch = 20  # Final fallback
        
        print(f"  Detected max epoch: {actual_max_epoch:.1f}")
        print(f"  Epoch-based metrics: {[tag for tag, _ in epoch_based_metrics]}")
        print(f"  Step-based metrics: {[tag for tag, _ in step_based_metrics]}")
        
        # Second pass: assign epochs appropriately
        for tag in tags:
            if len(dfs[tag]) > 0:
                max_step = dfs[tag]['step'].max()
                
                if any(tag == epoch_tag for epoch_tag, _ in epoch_based_metrics):
                    # Already epoch-based
                    dfs[tag]['epoch'] = dfs[tag]['step']
                    print(f"    {tag}: Using step as epoch (max: {max_step})")
                else:
                    # Convert from steps to epochs
                    steps_per_epoch = max_step / actual_max_epoch
                    dfs[tag]['epoch'] = dfs[tag]['step'] / steps_per_epoch
                    print(f"    {tag}: Converting steps to epochs (steps_per_epoch: {steps_per_epoch:.0f})")
        
        return dfs, tags
    
    def plot_training_curves(self, metrics=['val_loss', 'train_loss_epoch', 'val_pearson'], 
                        top_n=15, figsize=(18, 6)):
        """Plot training curves grouped by experiment type - FIXED"""
        fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
        if len(metrics) == 1:
            axes = [axes]
        
        # Group colors
        group_colors = {
            'Random Init LR Sweep': 'blue',
            'Human-Borzoi LR Sweep': 'red', 
            'Human-Borzoi Best LR (3e-5)': 'green'
        }
        
        # Group line styles
        group_linestyles = {
            'Random Init LR Sweep': '-',
            'Human-Borzoi LR Sweep': '--', 
            'Human-Borzoi Best LR (3e-5)': ':'
        }
        
        for ax, metric in zip(axes, metrics):
            # Group experiments by type FIRST
            grouped_experiments = {}
            for exp_name, data in self.experiment_data.items():
                group = data['params'].get('experiment_group', 'Unknown')
                if group not in grouped_experiments:
                    grouped_experiments[group] = []
                grouped_experiments[group].append((exp_name, data))
            
            # Plot each group with consistent styling
            for group, experiments in grouped_experiments.items():
                color = group_colors.get(group, 'gray')
                linestyle = group_linestyles.get(group, '-')
                alpha = 0.7
                
                plotted_for_group = 0
                for exp_name, data in experiments:
                    if plotted_for_group >= top_n // len(grouped_experiments):  # Distribute across groups
                        break
                    if metric in data['metrics'] and not data['metrics'][metric].empty:
                        df = data['metrics'][metric]
                        
                        # Plot with group-specific styling
                        ax.plot(df['epoch'], df['value'], 
                            color=color, alpha=alpha, linewidth=1.5,
                            linestyle=linestyle,
                            label=group if plotted_for_group == 0 else "_nolegend_")  # Only label first of each group
                        plotted_for_group += 1
            
            ax.set_title(metric.replace('_', ' ').title(), fontsize=12)
            ax.set_xlabel('Training Steps')
            ax.set_ylabel('Value')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9)
        
        plt.suptitle('Training Curves by Experiment Groups\n'
                    'Blue=Random Init, Red=Human-Borzoi LR Sweep, Green=Human-Borzoi Best LR', 
                    fontsize=14, y=1.02)
        plt.tight_layout()
        return fig

    def plot_min_val_loss_vs_lr(self, figsize=(14, 8), use_early_stopped=True):
        """Plot minimum validation loss vs learning rate with PROPER experiment groups"""
        data_points = []
        
        for exp_name, data in self.experiment_data.items():
            if 'lr' not in data['params']:
                continue
                
            # Use early-stopped minimum if available
            if use_early_stopped and 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                min_loss = data['metrics']['val_loss']['value'].min()
            else:
                continue
                
            data_points.append({
                'experiment': exp_name,
                'lr': data['params']['lr'],
                'min_val_loss': min_loss,
                'weight': data['params'].get('weight', 'unknown'),
                'model_run': data['params'].get('replicate', 'unknown'),
                'experiment_group': data['params'].get('experiment_group', 'Unknown')
            })
        
        if not data_points:
            print("No data available for min val loss vs lr plot")
            return
        
        df = pd.DataFrame(data_points)
        
        plt.figure(figsize=figsize)
        
        # Define colors and markers for each group
        group_styles = {
            'Random Init LR Sweep': {'color': 'blue', 'marker': 'o'},
            'Human-Borzoi LR Sweep': {'color': 'red', 'marker': 's'},
            'Human-Borzoi Best LR (3e-5)': {'color': 'green', 'marker': '^'}
        }
        
        # Plot each group separately with proper styling
        for group in df['experiment_group'].unique():
            group_df = df[df['experiment_group'] == group]
            style = group_styles.get(group, {'color': 'gray', 'marker': 'o'})
            
            if group == 'Human-Borzoi Best LR (3e-5)':
                # For the replicates, show different marker styles within the group
                for model_run in group_df['model_run'].unique():
                    run_df = group_df[group_df['model_run'] == model_run]
                    plt.scatter(run_df['lr'], run_df['min_val_loss'], 
                            c=style['color'], marker=style['marker'], s=120, 
                            alpha=0.8, edgecolors='black', linewidth=0.5,
                            label=f"{group} (run {model_run})")
            else:
                # For LR sweeps, use consistent styling
                plt.scatter(group_df['lr'], group_df['min_val_loss'], 
                        c=style['color'], marker=style['marker'], s=120, 
                        alpha=0.8, edgecolors='black', linewidth=0.5,
                        label=group)
        
        plt.xscale('log')
        plt.xlabel('Learning Rate (log scale)', fontsize=12)
        plt.ylabel('Minimum Validation Loss (Early Stopped)', fontsize=12)
        plt.title('Early-Stopped Minimum Validation Loss vs Learning Rate\nGrouped by Experiment Type', fontsize=14)
        plt.grid(True, alpha=0.3)
        
        # Custom legend
        plt.legend(title='Experiment Groups', title_fontsize=12, fontsize=10, 
                bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        return plt.gcf()

    def plot_val_loss_curves_with_markers(self, top_n=15, figsize=(14, 7)):
        """Plot validation loss curves with proper group coloring"""
        min_losses = []
        
        for exp_name, data in self.experiment_data.items():
            if 'val_loss' not in data['metrics']:
                continue
                
            val_loss_df = data['metrics']['val_loss']
            if val_loss_df.empty:
                continue
            
            # Use early-stopped minimum if available
            if 'val_loss_min_value' in data['metrics']:
                min_val = data['metrics']['val_loss_min_value']
                min_epoch = data['metrics'].get('val_loss_min_epoch', val_loss_df['epoch'].iloc[-1])
            else:
                min_val = val_loss_df['value'].min()
                min_idx = val_loss_df['value'].idxmin()
                min_epoch = val_loss_df.loc[min_idx, 'epoch']
            
            final_epoch = val_loss_df['epoch'].iloc[-1]
            
            min_losses.append({
                'experiment': exp_name,
                'min_val_loss': min_val,
                'min_epoch': min_epoch,
                'final_epoch': final_epoch,
                'experiment_group': data['params'].get('experiment_group', 'Unknown'),
                'params': data['params']
            })
        
        if not min_losses:
            print("No validation loss data available")
            return
        
        # Sort by minimum validation loss and take top N
        min_df = pd.DataFrame(min_losses).sort_values('min_val_loss', ascending=True)
        if top_n:
            min_df = min_df.head(top_n)
        
        # Group colors
        group_colors = {
            'Random Init LR Sweep': 'blue',
            'Human-Borzoi LR Sweep': 'red', 
            'Human-Borzoi Best LR (3e-5)': 'green'
        }
        
        plt.figure(figsize=figsize)
        
        # Group by experiment type and assign colors accordingly
        for i, (_, row) in enumerate(min_df.iterrows()):
            exp_name = row['experiment']
            group = row['experiment_group']
            base_color = group_colors.get(group, 'gray')
            
            val_loss_df = self.experiment_data[exp_name]['metrics']['val_loss']
            
            # Plot curve with group color
            plt.plot(val_loss_df['epoch'], val_loss_df['value'], 
                    color=base_color, label=f"{exp_name} [{group}]", 
                    alpha=0.7, linewidth=1.5)
            
            # Add marker at early-stopped minimum (star)
            plt.scatter(row['min_epoch'], row['min_val_loss'], 
                    color=base_color, s=150, marker='*', 
                    edgecolor='black', linewidth=1.5, zorder=10)
            
            # Add marker at final epoch (circle)
            final_val = val_loss_df['value'].iloc[-1]
            plt.scatter(row['final_epoch'], final_val, 
                    color=base_color, s=80, marker='o', 
                    edgecolor='black', linewidth=1, zorder=9, alpha=0.7)
        
        plt.title(f'Validation Loss Curves (Top {len(min_df)} Models)\n★ = Early-stopped best, ● = Training end\nBlue=Random Init, Red=Human-Borzoi LR Sweep, Green=Human-Borzoi Best LR')
        plt.xlabel('Training Steps')
        plt.ylabel('Validation Loss')
        plt.grid(True, alpha=0.3)
        
        # Create custom legend for groups only
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='blue', lw=2, label='Random Init LR Sweep'),
            Line2D([0], [0], color='red', lw=2, label='Human-Borzoi LR Sweep'),
            Line2D([0], [0], color='green', lw=2, label='Human-Borzoi Best LR (3e-5)')
        ]
        plt.legend(handles=legend_elements, title='Experiment Groups', 
                bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        return plt.gcf()


    def get_top_random_init_models(self, top_n=5):
        """Get the top N models by early-stopped validation loss from RANDOM INIT experiments only"""
        model_results = []
        
        for exp_name, data in self.experiment_data.items():
            # Only include Random Init experiments
            if data['params'].get('experiment_group') != 'Random Init LR Sweep':
                continue
                
            if 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
                min_epoch = data['metrics']['val_loss_min_epoch']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                val_loss_df = data['metrics']['val_loss']
                min_loss = val_loss_df['value'].min()
                min_idx = val_loss_df['value'].idxmin()
                min_epoch = val_loss_df.loc[min_idx, 'epoch']
            else:
                continue
            
            model_results.append({
                'experiment': exp_name,
                'min_val_loss': min_loss,
                'min_epoch': min_epoch,
                'path': data['path'],
                'experiment_group': data['params'].get('experiment_group', 'Unknown'),
                'params': data['params']
            })
        
        # Sort by minimum validation loss
        model_results.sort(key=lambda x: x['min_val_loss'])
        
        print(f"\n🏆 TOP {top_n} RANDOM INIT MODELS BY EARLY-STOPPED VALIDATION LOSS:")
        print("=" * 100)
        
        for i, model in enumerate(model_results[:top_n], 1):
            print(f"{i}. {model['experiment']} [{model['experiment_group']}]")
            print(f"   Min Val Loss: {model['min_val_loss']:.6f} (at epoch {model['min_epoch']:.1f})")
            print(f"   Path: {model['path']}")
            print(f"   Params: {model['params']}")
            print()
        
        return model_results[:top_n]
    
    def _get_distinct_colors(self, n):
        """Generate distinct colors"""
        if n <= 10:
            return sns.color_palette("tab10", n)
        
        colors = []
        for i in range(n):
            hue = i/n
            saturation = 0.7 + 0.3 * (i % 3) / 2
            value = 0.8 + 0.2 * ((i+1) % 2)
            rgb = colorsys.hsv_to_rgb(hue, saturation, value)
            colors.append(rgb)
        return colors

# Usage remains the same:
base_dirs = [
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136570", #random init varying lr
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19040183", #random init varying lr
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19136584", #human-borzoi across borzoi replicates, fixed lr
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_19039194" #human-borzoi varying lr

]

analyzer = ExperimentAnalyzer()
analyzer.load_multiple_dirs(base_dirs)

# Get and print top 5 models
top_models = analyzer.get_top_random_init_models(top_n=5)

# Create the plots you want
analyzer.plot_training_curves(['val_loss', 'train_loss_epoch', 'val_pearson'], top_n=50)
plt.show()

analyzer.plot_min_val_loss_vs_lr()
plt.show()

analyzer.plot_val_loss_curves_with_markers(top_n=50)
plt.show()

In [ ]:
# Get and print best Human-Borzoi replicate
def get_best_human_borzoi_replicate(analyzer):
    """Get the best Human-Borzoi replicate from the 4 models run at fixed LR (3e-5)"""
    model_results = []
    
    for exp_name, data in analyzer.experiment_data.items():
        # Only include Human-Borzoi Best LR experiments (the 4 replicates)
        if data['params'].get('experiment_group') != 'Human-Borzoi Best LR (3e-5)':
            continue
            
        if 'val_loss_min_value' in data['metrics']:
            min_loss = data['metrics']['val_loss_min_value']
            min_epoch = data['metrics']['val_loss_min_epoch']
        elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
            val_loss_df = data['metrics']['val_loss']
            min_loss = val_loss_df['value'].min()
            min_idx = val_loss_df['value'].idxmin()
            min_epoch = val_loss_df.loc[min_idx, 'epoch']
        else:
            continue
        
        model_results.append({
            'experiment': exp_name,
            'min_val_loss': min_loss,
            'min_epoch': min_epoch,
            'path': data['path'],
            'experiment_group': data['params'].get('experiment_group', 'Unknown'),
            'replicate': data['params'].get('replicate', 'unknown'),
            'params': data['params']
        })
    
    # Sort by minimum validation loss
    model_results.sort(key=lambda x: x['min_val_loss'])
    
    print(f"\n🏆 BEST HUMAN-BORZOI REPLICATE MODEL (out of {len(model_results)} replicates):")
    print("=" * 100)
    
    if model_results:
        best_model = model_results[0]
        print(f"🥇 WINNER: {best_model['experiment']} [Replicate {best_model['replicate']}]")
        print(f"   Min Val Loss: {best_model['min_val_loss']:.6f} (at epoch {best_model['min_epoch']:.1f})")
        print(f"   Path: {best_model['path']}")
        print(f"   Params: {best_model['params']}")
        print()
        
        print("📊 ALL HUMAN-BORZOI REPLICATES RANKED:")
        print("-" * 80)
        for i, model in enumerate(model_results, 1):
            emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "📍"
            print(f"{emoji} {i}. Replicate {model['replicate']}: Val Loss = {model['min_val_loss']:.6f}")
    else:
        print("No Human-Borzoi replicate models found!")
    
    return model_results[0] if model_results else None

# Call the function
best_human_borzoi = get_best_human_borzoi_replicate(analyzer)

In [ ]:
# Get and print best Human-Borzoi replicate by PEARSON CORRELATION
def get_best_human_borzoi_replicate_by_pearson(analyzer):
    """Get the best Human-Borzoi replicate from the 4 models run at fixed LR (3e-5) by PEARSON"""
    model_results = []
    
    for exp_name, data in analyzer.experiment_data.items():
        # Only include Human-Borzoi Best LR experiments (the 4 replicates)
        if data['params'].get('experiment_group') != 'Human-Borzoi Best LR (3e-5)':
            continue
        
        # Look for validation Pearson correlation
        max_pearson = None
        max_epoch = None
        
        # Check for early-stopped Pearson metrics first
        if 'val_pearson_max_value' in data['metrics']:
            max_pearson = data['metrics']['val_pearson_max_value']
            max_epoch = data['metrics']['val_pearson_max_epoch']
        elif 'val_pearson' in data['metrics'] and not data['metrics']['val_pearson'].empty:
            pearson_df = data['metrics']['val_pearson']
            max_pearson = pearson_df['value'].max()
            max_idx = pearson_df['value'].idxmax()
            max_epoch = pearson_df.loc[max_idx, 'epoch']
        else:
            # Try other possible Pearson metric names
            pearson_metrics = [metric for metric in data['metrics'].keys() if 'pearson' in metric.lower()]
            if pearson_metrics:
                for metric in pearson_metrics:
                    if not data['metrics'][metric].empty:
                        pearson_df = data['metrics'][metric]
                        max_pearson = pearson_df['value'].max()
                        max_idx = pearson_df['value'].idxmax()
                        max_epoch = pearson_df.loc[max_idx, 'epoch']
                        break
        
        if max_pearson is not None:
            model_results.append({
                'experiment': exp_name,
                'max_pearson': max_pearson,
                'max_epoch': max_epoch,
                'path': data['path'],
                'experiment_group': data['params'].get('experiment_group', 'Unknown'),
                'replicate': data['params'].get('replicate', 'unknown'),
                'params': data['params']
            })
    
    # Sort by maximum Pearson correlation (DESCENDING - higher is better)
    model_results.sort(key=lambda x: x['max_pearson'], reverse=True)
    
    print(f"\n🏆 BEST HUMAN-BORZOI REPLICATE MODEL BY PEARSON CORRELATION (out of {len(model_results)} replicates):")
    print("=" * 100)
    
    if model_results:
        best_model = model_results[0]
        print(f"🥇 WINNER: {best_model['experiment']} [Replicate {best_model['replicate']}]")
        print(f"   Max Pearson: {best_model['max_pearson']:.6f} (at epoch {best_model['max_epoch']:.1f})")
        print(f"   Path: {best_model['path']}")
        print(f"   Params: {best_model['params']}")
        print()
        
        print("📊 ALL HUMAN-BORZOI REPLICATES RANKED BY PEARSON:")
        print("-" * 80)
        for i, model in enumerate(model_results, 1):
            emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else "📍"
            print(f"{emoji} {i}. Replicate {model['replicate']}: Pearson = {model['max_pearson']:.6f}")
    else:
        print("No Human-Borzoi replicate models with Pearson data found!")
    
    return model_results[0] if model_results else None

# Call the function
best_human_borzoi_pearson = get_best_human_borzoi_replicate_by_pearson(analyzer)

### Inlcuding Human-Decima initializations, and re-runs of Human-Borzoi, Human-Mouse, and Random initializations. Val_loss_diff for early stopping at 0.0001.
### Figure 1 in the DanioDecima manuscript

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorboard.backend.event_processing import event_accumulator
import glob
import re
import colorsys

class ExperimentAnalyzer:
    def __init__(self):
        self.experiment_data = {}
        self.all_metrics = set()
    
    def load_multiple_dirs(self, base_dirs):
        """Load experiments from multiple base directories"""
        all_experiment_dirs = []
        for base_dir in base_dirs:
            dirs = self._find_experiment_dirs(base_dir)
            # Attach base_dir to each experiment
            all_experiment_dirs.extend([(d, base_dir) for d in dirs])
        
        print(f"Total experiment directories found: {len(all_experiment_dirs)}")
        self.experiment_data, self.all_metrics = self._load_experiment_data(all_experiment_dirs)
        print(f"Loaded {len(self.experiment_data)} experiments with metrics: {sorted(self.all_metrics)}")
        
        # Print summary by source directory
        self._print_loading_summary()
        
    def _print_loading_summary(self):
        """Print summary of loaded experiments by source directory"""
        print("\n" + "="*80)
        print("EXPERIMENT LOADING SUMMARY")
        print("="*80)
        
        source_counts = {}
        for exp_name, data in self.experiment_data.items():
            source_dir = data['params'].get('source_dir', 'Unknown')
            source_name = self._get_source_name(source_dir)
            if source_name not in source_counts:
                source_counts[source_name] = 0
            source_counts[source_name] += 1
        
        for source_name, count in source_counts.items():
            expected = self._get_expected_count(source_name)
            status = "✓" if count == expected else "⚠️"
            print(f"{status} {source_name}: {count}/{expected} experiments loaded")
        
        print(f"\nTotal: {len(self.experiment_data)} experiments loaded")
        print("="*80)
    
    def _get_source_name(self, source_dir):
        """Extract readable name from source directory"""
        if 'decima_tune_20530488' in source_dir:
            return 'Random Init (3e-6)'
        elif 'decima_tune_20581454' in source_dir:
            return 'Mouse-Borzoi (3e-5)'
        elif 'decima_tune_20586902' in source_dir:
            return 'Human-Borzoi (3e-5)'
        elif any(tune_id in source_dir for tune_id in ['20473753', '20491529', '20491962', '20491359']):
            return 'Human-Decima (3e-5)'
        else:
            return os.path.basename(source_dir)
    
    def _get_expected_count(self, source_name):
        """Get expected number of experiments for each source"""
        if source_name in ['Random Init (3e-6)', 'Mouse-Borzoi (3e-5)', 'Human-Borzoi (3e-5)']:
            return 4  # 4x replicates
        elif source_name == 'Human-Decima (3e-5)':
            return 4  # 4 separate directories with 1 experiment each
        else:
            return 1  # default
        
    def _find_experiment_dirs(self, base_dir):
        """Find all experiment directories - fixed to match your original logic"""
        array_dirs = glob.glob(os.path.join(base_dir, "*"))
        experiment_dirs = []
        
        print(f"Scanning for experiments in {base_dir}...")
        
        for array_dir in array_dirs:
            if not os.path.isdir(array_dir):
                continue
                
            # Look for task directories
            task_dirs = glob.glob(os.path.join(array_dir, "task_*"))
            
            if task_dirs:  # Ray Tune structure
                for task_dir in task_dirs:
                    print(f"  Checking task directory: {task_dir}")
                    
                    # Find trial directories (hyperparameter combinations)
                    trial_dirs = glob.glob(os.path.join(task_dir, "lr_*")) + \
                               glob.glob(os.path.join(task_dir, "train_*"))
                    
                    print(f"    Found {len(trial_dirs)} trial directories")
                    
                    for trial_dir in trial_dirs:
                        print(f"      Checking trial: {trial_dir}")
                        
                        # Look for version directories or directly for event files
                        version_dirs = glob.glob(os.path.join(trial_dir, "version_*"))
                        
                        if version_dirs:
                            for version_dir in version_dirs:
                                experiment_dirs.append(version_dir)
                                print(f"        Added version dir: {version_dir}")
                        else:
                            if glob.glob(os.path.join(trial_dir, "events.out.tfevents.*")):
                                experiment_dirs.append(trial_dir)
                                print(f"        Added trial dir: {trial_dir}")
            else:
                # Check if the array directory itself contains event files
                event_files = glob.glob(os.path.join(array_dir, "events.out.tfevents.*"))
                if event_files:
                    experiment_dirs.append(array_dir)
                    print(f"  Added array dir: {array_dir}")
        
        print(f"Found {len(experiment_dirs)} total experiment directories")
        return experiment_dirs
    
    def _calculate_early_stopping_metrics(self, df, patience=10, epsilon=0.001, minimize=True):
        """
        Calculate early stopping metrics properly using patience and epsilon
        """
        if df.empty:
            return None
        
        # Sort by epoch to ensure chronological order
        df_sorted = df.sort_values('epoch').reset_index(drop=True)
        
        best_value = None
        best_epoch = None
        best_step = None
        patience_counter = 0
        early_stop_epoch = None
        
        for idx, row in df_sorted.iterrows():
            current_value = row['value']
            current_epoch = row['epoch']
            current_step = row['step']
            
            # Check if this is an improvement
            is_improvement = False
            if best_value is None:
                is_improvement = True
            else:
                if minimize:
                    is_improvement = (best_value - current_value) > epsilon
                else:
                    is_improvement = (current_value - best_value) > epsilon
            
            if is_improvement:
                best_value = current_value
                best_epoch = current_epoch
                best_step = current_step
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Check if we should stop
            if patience_counter >= patience:
                early_stop_epoch = current_epoch
                break
        
        return {
            'best_value': best_value,
            'best_epoch': best_epoch,
            'best_step': best_step,
            'early_stop_epoch': early_stop_epoch,
            'patience_exhausted': patience_counter >= patience
        }

    def _extract_experiment_group(self, path, source_dir):
        """Extract experiment group from directory path and source directory"""
        source_name = self._get_source_name(source_dir)
        
        # Map source names to experiment groups
        if source_name == 'Random Init (3e-6)':
            return 'Random Init (3e-6)'
        elif source_name == 'Mouse-Borzoi (3e-5)':
            return 'Mouse-Borzoi (3e-5)'
        elif source_name == 'Human-Borzoi (3e-5)':
            return 'Human-Borzoi (3e-5)'
        elif source_name == 'Human-Decima (3e-5)':
            return 'Human-Decima (3e-5)'
        else:
            return 'Unknown'

    def _load_experiment_data(self, experiment_dirs):
        """Load experiment data with group information"""
        experiment_data = {}
        all_metrics = set()
        
        for exp_dir, base_dir in experiment_dirs:
            event_files = glob.glob(os.path.join(exp_dir, "events.out.tfevents.*"))
            
            if not event_files:
                continue
            
            params = self._extract_params_from_path(exp_dir)
            params['source_dir'] = base_dir
            
            # Add experiment group based on source directory
            exp_group = self._extract_experiment_group(exp_dir, base_dir)
            params['experiment_group'] = exp_group
            
            # Create descriptive name
            source_name = self._get_source_name(base_dir)
            if 'replicate' in params:
                desc_name = f"{source_name}_rep={params['replicate']}"
            else:
                desc_name = f"{source_name}_{os.path.basename(exp_dir)}"
            
            latest_event = sorted(event_files, key=os.path.getmtime)[-1]
            try:
                metrics_data, metrics = self._load_tb_file(latest_event)
                
                # Apply early stopping analysis
                early_stopping_patience = 10
                early_stopping_epsilon = 0.001
                
                for metric in metrics:
                    if metric in metrics_data:
                        df = metrics_data[metric]
                        if df.empty:
                            continue
                        
                        if 'loss' in metric.lower():
                            es_result = self._calculate_early_stopping_metrics(
                                df, patience=early_stopping_patience, 
                                epsilon=early_stopping_epsilon, minimize=True
                            )
                            
                            if es_result:
                                metrics_data[f"{metric}_early_stop"] = es_result
                                metrics_data[f"{metric}_min_value"] = es_result['best_value']
                                metrics_data[f"{metric}_min_step"] = es_result['best_step']
                                metrics_data[f"{metric}_min_epoch"] = es_result['best_epoch']
                
                all_metrics.update(metrics)
                
                experiment_data[desc_name] = {
                    'path': exp_dir,
                    'params': params,
                    'metrics': metrics_data
                }
                print(f"  Loaded {desc_name} [{exp_group}]")
            except Exception as e:
                print(f"  ERROR loading {latest_event}: {e}")
        
        return experiment_data, list(all_metrics)
    
    def _extract_params_from_path(self, path):
        """Extract parameters including replicate information"""
        params = {}
        
        # Extract learning rate
        lr_patterns = [r'lr[_=]([\d.e\-]+)', r'lr_([\d.e\-]+)']
        for pattern in lr_patterns:
            lr_match = re.search(pattern, path)
            if lr_match:
                params['lr'] = float(lr_match.group(1))
                break
        
        # Extract batch size
        bs_patterns = [r'bs[_=](\d+)', r'bs_(\d+)']
        for pattern in bs_patterns:
            bs_match = re.search(pattern, path)
            if bs_match:
                params['bs'] = int(bs_match.group(1))
                break
        
        # Extract weight parameter
        w_patterns = [r'w[_=]([\d.e\-]+)', r'w_([\d.e\-]+)']
        for pattern in w_patterns:
            w_match = re.search(pattern, path)
            if w_match:
                params['weight'] = float(w_match.group(1))
                break
        
        # Extract replicate information - prioritize task numbers for Ray Tune
        rep_patterns = [
            r'task_(\d+)',     # This should catch task_0, task_1, etc.
            r'rep[_=](\d+)', 
            r'rep_(\d+)', 
            r'replicate[_=](\d+)',
            r'version_(\d+)',  # version directories as fallback
            r'trial_(\d+)',    # trial numbers
            r'run_(\d+)'       # run numbers
        ]
        for pattern in rep_patterns:
            rep_match = re.search(pattern, path)
            if rep_match:
                params['replicate'] = int(rep_match.group(1))
                break
        
        return params

    def _load_tb_file(self, path):
        """Load tensorboard file with adaptive epoch handling"""
        ea = event_accumulator.EventAccumulator(path, size_guidance={'scalars': 0})
        ea.Reload()
        
        tags = ea.Tags()['scalars']
        dfs = {}
        
        # First pass: load all data and find epoch-based metrics
        epoch_based_metrics = []
        step_based_metrics = []
        actual_max_epoch = None
        
        for tag in tags:
            events = ea.Scalars(tag)
            dfs[tag] = pd.DataFrame([
                {'step': e.step, 'value': e.value, 'wall_time': e.wall_time}
                for e in events
            ])
            
            if len(dfs[tag]) > 0:
                max_step = dfs[tag]['step'].max()
                
                # Classify metrics by their characteristics
                if 'epoch' in tag.lower():
                    epoch_based_metrics.append((tag, max_step))
                    if actual_max_epoch is None or max_step > actual_max_epoch:
                        actual_max_epoch = max_step
                elif max_step <= 50:  # Heuristic: small numbers are likely epochs
                    epoch_based_metrics.append((tag, max_step))
                    if actual_max_epoch is None or max_step > actual_max_epoch:
                        actual_max_epoch = max_step
                else:
                    step_based_metrics.append((tag, max_step))
        
        # If we found epoch-based metrics, use them to determine the actual max epoch
        if actual_max_epoch is None:
            if step_based_metrics:
                estimated_steps_per_epoch = 1000  # Adjust this for your dataset
                max_steps = max(max_step for _, max_step in step_based_metrics)
                actual_max_epoch = max_steps / estimated_steps_per_epoch
            else:
                actual_max_epoch = 20  # Final fallback
        
        print(f"  Detected max epoch: {actual_max_epoch:.1f}")
        
        # Second pass: assign epochs appropriately
        for tag in tags:
            if len(dfs[tag]) > 0:
                max_step = dfs[tag]['step'].max()
                
                if any(tag == epoch_tag for epoch_tag, _ in epoch_based_metrics):
                    dfs[tag]['epoch'] = dfs[tag]['step']
                else:
                    steps_per_epoch = max_step / actual_max_epoch
                    dfs[tag]['epoch'] = dfs[tag]['step'] / steps_per_epoch
        
        return dfs, tags
    
    def plot_training_curves(self, metrics=['val_loss', 'train_loss_epoch', 'val_pearson'], 
                        top_n=15, figsize=(18, 6)):
        """Plot training curves grouped by experiment type"""
        fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
        if len(metrics) == 1:
            axes = [axes]
        
        # Group colors - updated for new experiment types
        group_colors = {
            'Random Init (3e-6)': 'blue',
            'Mouse-Borzoi (3e-5)': 'red', 
            'Human-Borzoi (3e-5)': 'green',
            'Human-Decima (3e-5)': 'purple'
        }
        
        # Group line styles
        group_linestyles = {
            'Random Init (3e-6)': '-',
            'Mouse-Borzoi (3e-5)': '--', 
            'Human-Borzoi (3e-5)': '-.',
            'Human-Decima (3e-5)': ':'
        }
        
        for ax, metric in zip(axes, metrics):
            # Group experiments by type
            grouped_experiments = {}
            for exp_name, data in self.experiment_data.items():
                group = data['params'].get('experiment_group', 'Unknown')
                if group not in grouped_experiments:
                    grouped_experiments[group] = []
                grouped_experiments[group].append((exp_name, data))
            
            # Plot each group with consistent styling
            for group, experiments in grouped_experiments.items():
                color = group_colors.get(group, 'gray')
                linestyle = group_linestyles.get(group, '-')
                alpha = 0.7
                
                plotted_for_group = 0
                for exp_name, data in experiments:
                    if plotted_for_group >= top_n // len(grouped_experiments):
                        break
                    if metric in data['metrics'] and not data['metrics'][metric].empty:
                        df = data['metrics'][metric]
                        
                        ax.plot(df['epoch'], df['value'], 
                            color=color, alpha=alpha, linewidth=1.5,
                            linestyle=linestyle,
                            label=group if plotted_for_group == 0 else "_nolegend_")
                        plotted_for_group += 1
            
            ax.set_title(metric.replace('_', ' ').title(), fontsize=12)
            ax.set_xlabel('Training Steps')
            ax.set_ylabel('Value')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=9)
        
        plt.suptitle('Training Curves by Experiment Groups', fontsize=14, y=1.02)
        plt.tight_layout()
        return fig

    def plot_min_val_loss_vs_lr(self, figsize=(14, 8), use_early_stopped=True):
        """Plot minimum validation loss vs learning rate with experiment groups"""
        data_points = []
        
        for exp_name, data in self.experiment_data.items():
            if 'lr' not in data['params']:
                continue
                
            # Use early-stopped minimum if available
            if use_early_stopped and 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                min_loss = data['metrics']['val_loss']['value'].min()
            else:
                continue
                
            data_points.append({
                'experiment': exp_name,
                'lr': data['params']['lr'],
                'min_val_loss': min_loss,
                'model_run': data['params'].get('replicate', 'unknown'),
                'experiment_group': data['params'].get('experiment_group', 'Unknown'),
                'source_dir': data['params'].get('source_dir', 'Unknown')
            })
        
        if not data_points:
            print("No data available for min val loss vs lr plot")
            return
        
        df = pd.DataFrame(data_points)
        
        plt.figure(figsize=figsize)
        
        # Define colors and markers for each group
        group_styles = {
            'Random Init (3e-6)': {'color': 'blue', 'marker': 'o'},
            'Mouse-Borzoi (3e-5)': {'color': 'red', 'marker': 's'},
            'Human-Borzoi (3e-5)': {'color': 'green', 'marker': '^'},
            'Human-Decima (3e-5)': {'color': 'purple', 'marker': 'D'}
        }
        
        # Plot each group separately
        for group in df['experiment_group'].unique():
            group_df = df[df['experiment_group'] == group]
            style = group_styles.get(group, {'color': 'gray', 'marker': 'o'})
            
            plt.scatter(group_df['lr'], group_df['min_val_loss'], 
                    c=style['color'], marker=style['marker'], s=120, 
                    alpha=0.8, edgecolors='black', linewidth=0.5,
                    label=f"{group} (n={len(group_df)})")
        
        plt.xscale('log')
        plt.xlabel('Learning Rate (log scale)', fontsize=12)
        plt.ylabel('Minimum Validation Loss (Early Stopped)', fontsize=12)
        plt.title('Early-Stopped Minimum Validation Loss vs Learning Rate\nGrouped by Experiment Type', fontsize=14)
        plt.grid(True, alpha=0.3)
        
        plt.legend(title='Experiment Groups', title_fontsize=12, fontsize=10, 
                bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        return plt.gcf()

    def plot_val_loss_curves_with_markers(self, top_n=15, figsize=(14, 7)):
        """Plot validation loss curves with proper group coloring"""
        min_losses = []
        
        for exp_name, data in self.experiment_data.items():
            if 'val_loss' not in data['metrics']:
                continue
                
            val_loss_df = data['metrics']['val_loss']
            if val_loss_df.empty:
                continue
            
            # Use early-stopped minimum if available
            if 'val_loss_min_value' in data['metrics']:
                min_val = data['metrics']['val_loss_min_value']
                min_epoch = data['metrics'].get('val_loss_min_epoch', val_loss_df['epoch'].iloc[-1])
            else:
                min_val = val_loss_df['value'].min()
                min_idx = val_loss_df['value'].idxmin()
                min_epoch = val_loss_df.loc[min_idx, 'epoch']
            
            final_epoch = val_loss_df['epoch'].iloc[-1]
            
            min_losses.append({
                'experiment': exp_name,
                'min_val_loss': min_val,
                'min_epoch': min_epoch,
                'final_epoch': final_epoch,
                'experiment_group': data['params'].get('experiment_group', 'Unknown'),
                'params': data['params']
            })
        
        if not min_losses:
            print("No validation loss data available")
            return
        
        # Sort by minimum validation loss and take top N
        min_df = pd.DataFrame(min_losses).sort_values('min_val_loss', ascending=True)
        if top_n:
            min_df = min_df.head(top_n)
        
        # Group colors
        group_colors = {
            'Random Init (3e-6)': 'blue',
            'Mouse-Borzoi (3e-5)': 'red', 
            'Human-Borzoi (3e-5)': 'green',
            'Human-Decima (3e-5)': 'purple'
        }
        
        plt.figure(figsize=figsize)
        
        # Group by experiment type and assign colors accordingly
        for i, (_, row) in enumerate(min_df.iterrows()):
            exp_name = row['experiment']
            group = row['experiment_group']
            base_color = group_colors.get(group, 'gray')
            
            val_loss_df = self.experiment_data[exp_name]['metrics']['val_loss']
            
            # Plot curve with group color
            plt.plot(val_loss_df['epoch'], val_loss_df['value'], 
                    color=base_color, alpha=0.7, linewidth=1.5)
            
            # Add marker at early-stopped minimum (star)
            plt.scatter(row['min_epoch'], row['min_val_loss'], 
                    color=base_color, s=150, marker='*', 
                    edgecolor='black', linewidth=1.5, zorder=10)
            
            # Add marker at final epoch (circle)
            final_val = val_loss_df['value'].iloc[-1]
            plt.scatter(row['final_epoch'], final_val, 
                    color=base_color, s=80, marker='o', 
                    edgecolor='black', linewidth=1, zorder=9, alpha=0.7)
        
        plt.title(f'Validation Loss Curves (Top {len(min_df)} Models)\n★ = Early-stopped best, ● = Training end')
        plt.xlabel('Training Steps')
        plt.ylabel('Validation Loss')
        plt.grid(True, alpha=0.3)
        
        # Create custom legend for groups only
        from matplotlib.lines import Line2D
        legend_elements = [
            Line2D([0], [0], color='blue', lw=2, label='Random Init (3e-6)'),
            Line2D([0], [0], color='red', lw=2, label='Mouse-Borzoi (3e-5)'),
            Line2D([0], [0], color='green', lw=2, label='Human-Borzoi (3e-5)'),
            Line2D([0], [0], color='purple', lw=2, label='Human-Decima (3e-5)')
        ]
        plt.legend(handles=legend_elements, title='Experiment Groups', 
                bbox_to_anchor=(1.05, 1), loc='upper left')
        
        plt.tight_layout()
        return plt.gcf()

    def get_top_models_by_group(self, top_n=5):
        """Get the top N models by early-stopped validation loss for each group"""
        model_results = {}
        
        for exp_name, data in self.experiment_data.items():
            group = data['params'].get('experiment_group', 'Unknown')
            
            if 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
                min_epoch = data['metrics']['val_loss_min_epoch']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                val_loss_df = data['metrics']['val_loss']
                min_loss = val_loss_df['value'].min()
                min_idx = val_loss_df['value'].idxmin()
                min_epoch = val_loss_df.loc[min_idx, 'epoch']
            else:
                continue
            
            if group not in model_results:
                model_results[group] = []
            
            model_results[group].append({
                'experiment': exp_name,
                'min_val_loss': min_loss,
                'min_epoch': min_epoch,
                'path': data['path'],
                'params': data['params']
            })
        
        # Sort each group by minimum validation loss
        for group in model_results:
            model_results[group].sort(key=lambda x: x['min_val_loss'])
        
        print(f"\n🏆 TOP {top_n} MODELS BY GROUP (EARLY-STOPPED VALIDATION LOSS):")
        print("=" * 100)
        
        for group, models in model_results.items():
            print(f"\n{group.upper()}:")
            print("-" * 50)
            for i, model in enumerate(models[:top_n], 1):
                print(f"{i}. {model['experiment']}")
                print(f"   Min Val Loss: {model['min_val_loss']:.6f} (at epoch {model['min_epoch']:.1f})")
                print(f"   Path: {model['path']}")
                print()
        
        return model_results

    def _get_distinct_colors(self, n):
        """Generate distinct colors"""
        if n <= 10:
            return sns.color_palette("tab10", n)
        
        colors = []
        for i in range(n):
            hue = i/n
            saturation = 0.7 + 0.3 * (i % 3) / 2
            value = 0.8 + 0.2 * ((i+1) % 2)
            rgb = colorsys.hsv_to_rgb(hue, saturation, value)
            colors.append(rgb)
        return colors

# Expected experiments:
# - decima_tune_20530488: Random Init (3e-6) - 4 experiments
# - decima_tune_20581454: Mouse-Borzoi (3e-5) - 4 experiments  
# - decima_tune_20586902: Human-Borzoi (3e-5) - 4 experiments
# - decima_tune_20473753: Human-Decima (3e-5) - 1 experiment
# - decima_tune_20491529: Human-Decima (3e-5) - 1 experiment
# - decima_tune_20491962: Human-Decima (3e-5) - 1 experiment
# - decima_tune_20491359: Human-Decima (3e-5) - 1 experiment
# Total expected: 16 experiments

base_dirs = [
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20530488", #random init @ lr =  3e-6, 4x
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20581454", #mouse-borzoi @ lr =  3e-5, 4x
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20586902", #human-borzoi @ lr =  3e-5, 4x
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20473753", #human-decima @ lr =  3e-5, 1x
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491529", #human-decima @ lr =  3e-5, 1x
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491962", #human-decima @ lr =  3e-5, 1x
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/ray_results/logs/decima_tune_20491359" #human-decima @ lr =  3e-5, 1x
]

analyzer = ExperimentAnalyzer()
analyzer.load_multiple_dirs(base_dirs)

# Get and print top models by group
top_models_by_group = analyzer.get_top_models_by_group(top_n=5)

# Create the plots
analyzer.plot_training_curves(['val_loss', 'train_loss_epoch', 'val_pearson'], top_n=16)
plt.show()

analyzer.plot_min_val_loss_vs_lr()
plt.show()

analyzer.plot_val_loss_curves_with_markers(top_n=16)
plt.show()

### Loss curves of models 2025-06-22

###

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorboard.backend.event_processing import event_accumulator
import glob
import json
import re

class NewExperimentAnalyzer:
    def __init__(self):
        self.experiment_data = {}
        self.all_metrics = set()
    
    def load_experiment_dirs(self, base_dirs):
        """Load experiments from the two specific directories"""
        all_experiment_dirs = []
        
        for base_dir in base_dirs:
            print(f"Scanning {base_dir}...")
            
            # Find all experiment subdirectories
            exp_dirs = glob.glob(os.path.join(base_dir, "*"))
            exp_dirs = [d for d in exp_dirs if os.path.isdir(d)]
            
            print(f"  Found {len(exp_dirs)} experiment directories")
            
            for exp_dir in exp_dirs:
                exp_name = os.path.basename(exp_dir)
                print(f"    Checking {exp_name}")
                
                # Look for version directories or direct tensorboard files
                version_dirs = glob.glob(os.path.join(exp_dir, "version_*"))
                
                if version_dirs:
                    # Has version directories - check each one
                    for version_dir in version_dirs:
                        event_files = glob.glob(os.path.join(version_dir, "events.out.tfevents.*"))
                        if event_files:
                            all_experiment_dirs.append((version_dir, base_dir, exp_dir))
                            print(f"      Added {os.path.basename(version_dir)}")
                else:
                    # Check if experiment directory has direct event files
                    event_files = glob.glob(os.path.join(exp_dir, "events.out.tfevents.*"))
                    if event_files:
                        all_experiment_dirs.append((exp_dir, base_dir, exp_dir))
                        print(f"      Added {exp_name}")
        
        print(f"\nTotal experiment directories found: {len(all_experiment_dirs)}")
        self.experiment_data, self.all_metrics = self._load_experiment_data(all_experiment_dirs)
        print(f"Loaded {len(self.experiment_data)} experiments with metrics: {sorted(self.all_metrics)}")
        
        # Print summary
        self._print_loading_summary()
    
    def _load_experiment_info(self, exp_base_dir):
        """Load experiment_info.json if it exists"""
        info_file = os.path.join(exp_base_dir, "experiment_info.json")
        if os.path.exists(info_file):
            try:
                with open(info_file, 'r') as f:
                    return json.load(f)
            except Exception as e:
                print(f"    Warning: Could not load {info_file}: {e}")
        return {}
    
    def _extract_experiment_params(self, exp_dir_name, exp_info):
        """Extract experiment parameters from directory name and experiment_info.json"""
        params = {}
        
        # Parse directory name for parameters
        # Examples: pretrained_wandb-human_rep0_lr3e-05_seed42, random_lr3e-06_seed42
        
        # Extract experiment type
        if exp_dir_name.startswith('pretrained_'):
            params['init_mode'] = 'pretrained'
            # Extract pretrained source
            if 'wandb-human' in exp_dir_name:
                params['pretrained_source'] = 'wandb-human'
                params['experiment_type'] = 'Human-Borzoi'
            elif 'wandb-mouse' in exp_dir_name:
                params['pretrained_source'] = 'wandb-mouse'
                params['experiment_type'] = 'Mouse-Borzoi'
            elif 'decima-human' in exp_dir_name:
                params['pretrained_source'] = 'decima-human'
                params['experiment_type'] = 'Human-Decima'
            else:
                params['pretrained_source'] = 'unknown'
                params['experiment_type'] = 'Unknown-Pretrained'
        elif exp_dir_name.startswith('random_'):
            params['init_mode'] = 'random'
            params['experiment_type'] = 'Random-Init'
        else:
            params['init_mode'] = 'unknown'
            params['experiment_type'] = 'Unknown'
        
        # Extract replicate
        rep_match = re.search(r'rep(\d+)', exp_dir_name)
        if rep_match:
            params['replicate'] = int(rep_match.group(1))
        
        # Extract learning rate
        lr_match = re.search(r'lr([\d.e\-]+)', exp_dir_name)
        if lr_match:
            params['lr'] = float(lr_match.group(1))
        
        # Extract seed
        seed_match = re.search(r'seed(\d+)', exp_dir_name)
        if seed_match:
            params['seed'] = int(seed_match.group(1))
        
        # Add info from experiment_info.json if available
        if exp_info:
            params.update(exp_info)
        
        return params
    
    def _calculate_early_stopping_metrics(self, df, patience=10, epsilon=0.001, minimize=True):
        """Calculate early stopping metrics"""
        if df.empty:
            return None
        
        df_sorted = df.sort_values('epoch').reset_index(drop=True)
        
        best_value = None
        best_epoch = None
        best_step = None
        patience_counter = 0
        early_stop_epoch = None
        
        for idx, row in df_sorted.iterrows():
            current_value = row['value']
            current_epoch = row['epoch']
            current_step = row['step']
            
            is_improvement = False
            if best_value is None:
                is_improvement = True
            else:
                if minimize:
                    is_improvement = (best_value - current_value) > epsilon
                else:
                    is_improvement = (current_value - best_value) > epsilon
            
            if is_improvement:
                best_value = current_value
                best_epoch = current_epoch
                best_step = current_step
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                early_stop_epoch = current_epoch
                break
        
        return {
            'best_value': best_value,
            'best_epoch': best_epoch,
            'best_step': best_step,
            'early_stop_epoch': early_stop_epoch,
            'patience_exhausted': patience_counter >= patience
        }
    
    def _load_experiment_data(self, experiment_dirs):
        """Load experiment data from directories"""
        experiment_data = {}
        all_metrics = set()
        
        for exp_dir, base_dir, exp_base_dir in experiment_dirs:
            # Get experiment name from the base experiment directory
            exp_name = os.path.basename(exp_base_dir)
            version_name = os.path.basename(exp_dir)
            
            # Load experiment info
            exp_info = self._load_experiment_info(exp_base_dir)
            
            # Extract parameters
            params = self._extract_experiment_params(exp_name, exp_info)
            params['base_dir'] = base_dir
            params['version'] = version_name
            
            # Find tensorboard files
            event_files = glob.glob(os.path.join(exp_dir, "events.out.tfevents.*"))
            if not event_files:
                continue
            
            # Load latest tensorboard file
            latest_event = sorted(event_files, key=os.path.getmtime)[-1]
            
            try:
                metrics_data, metrics = self._load_tb_file(latest_event)
                
                # Apply early stopping analysis
                for metric in metrics:
                    if metric in metrics_data and not metrics_data[metric].empty:
                        if 'loss' in metric.lower():
                            es_result = self._calculate_early_stopping_metrics(
                                metrics_data[metric], patience=10, epsilon=0.001, minimize=True
                            )
                            if es_result:
                                metrics_data[f"{metric}_early_stop"] = es_result
                                metrics_data[f"{metric}_min_value"] = es_result['best_value']
                                metrics_data[f"{metric}_min_step"] = es_result['best_step']
                                metrics_data[f"{metric}_min_epoch"] = es_result['best_epoch']
                        elif 'pearson' in metric.lower():
                            es_result = self._calculate_early_stopping_metrics(
                                metrics_data[metric], patience=10, epsilon=0.001, minimize=False
                            )
                            if es_result:
                                metrics_data[f"{metric}_early_stop"] = es_result
                                metrics_data[f"{metric}_max_value"] = es_result['best_value']
                                metrics_data[f"{metric}_max_step"] = es_result['best_step']
                                metrics_data[f"{metric}_max_epoch"] = es_result['best_epoch']
                
                all_metrics.update(metrics)
                
                # Create unique experiment name
                if version_name != exp_name:
                    full_name = f"{exp_name}_{version_name}"
                else:
                    full_name = exp_name
                
                experiment_data[full_name] = {
                    'path': exp_dir,
                    'params': params,
                    'metrics': metrics_data
                }
                
                print(f"  Loaded {full_name} [{params['experiment_type']}]")
                
            except Exception as e:
                print(f"  ERROR loading {latest_event}: {e}")
        
        return experiment_data, list(all_metrics)
    
    def _load_tb_file(self, path):
        """Load tensorboard file"""
        ea = event_accumulator.EventAccumulator(path, size_guidance={'scalars': 0})
        ea.Reload()
        
        tags = ea.Tags()['scalars']
        dfs = {}
        
        # Load all metrics
        for tag in tags:
            events = ea.Scalars(tag)
            dfs[tag] = pd.DataFrame([
                {'step': e.step, 'value': e.value, 'wall_time': e.wall_time}
                for e in events
            ])
            
            if len(dfs[tag]) > 0:
                # Simple epoch calculation - assume step-based metrics
                max_step = dfs[tag]['step'].max()
                if max_step <= 50:  # Likely already epochs
                    dfs[tag]['epoch'] = dfs[tag]['step']
                else:
                    # Estimate epochs (adjust based on your training setup)
                    steps_per_epoch = 1000  # Adjust this value
                    dfs[tag]['epoch'] = dfs[tag]['step'] / steps_per_epoch
        
        return dfs, tags
    
    def _print_loading_summary(self):
        """Print summary of loaded experiments"""
        print("\n" + "="*80)
        print("EXPERIMENT LOADING SUMMARY")
        print("="*80)
        
        type_counts = {}
        for exp_name, data in self.experiment_data.items():
            exp_type = data['params'].get('experiment_type', 'Unknown')
            if exp_type not in type_counts:
                type_counts[exp_type] = 0
            type_counts[exp_type] += 1
        
        for exp_type, count in sorted(type_counts.items()):
            print(f"  {exp_type}: {count} experiments")
        
        print(f"\nTotal: {len(self.experiment_data)} experiments loaded")
        print("="*80)
    
    def plot_training_curves(self, metrics=['val_loss', 'train_loss_epoch', 'val_pearson'], 
                           figsize=(18, 6)):
        """Plot training curves grouped by experiment type"""
        fig, axes = plt.subplots(1, len(metrics), figsize=figsize)
        if len(metrics) == 1:
            axes = [axes]
        
        # Define colors for each experiment type
        colors = {
            'Human-Borzoi': 'green',
            'Mouse-Borzoi': 'red',
            'Human-Decima': 'purple',
            'Random-Init': 'blue'
        }
        
        for ax, metric in zip(axes, metrics):
            # Group experiments by type
            grouped_experiments = {}
            for exp_name, data in self.experiment_data.items():
                exp_type = data['params'].get('experiment_type', 'Unknown')
                if exp_type not in grouped_experiments:
                    grouped_experiments[exp_type] = []
                grouped_experiments[exp_type].append((exp_name, data))
            
            # Plot each group
            for exp_type, experiments in grouped_experiments.items():
                color = colors.get(exp_type, 'gray')
                
                for i, (exp_name, data) in enumerate(experiments):
                    if metric in data['metrics'] and not data['metrics'][metric].empty:
                        df = data['metrics'][metric]
                        label = exp_type if i == 0 else "_nolegend_"
                        ax.plot(df['epoch'], df['value'], 
                               color=color, alpha=0.7, linewidth=1.5, label=label)
            
            ax.set_title(metric.replace('_', ' ').title(), fontsize=12)
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Value')
            ax.grid(True, alpha=0.3)
            ax.legend()
        
        plt.suptitle('Training Curves by Experiment Type', fontsize=14)
        plt.tight_layout()
        return fig
    
    def plot_min_val_loss_vs_lr(self, figsize=(12, 8)):
        """Plot minimum validation loss vs learning rate"""
        data_points = []
        
        for exp_name, data in self.experiment_data.items():
            params = data['params']
            
            # Get minimum validation loss
            if 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                min_loss = data['metrics']['val_loss']['value'].min()
            else:
                continue
            
            data_points.append({
                'experiment': exp_name,
                'lr': params.get('lr', 0),
                'min_val_loss': min_loss,
                'experiment_type': params.get('experiment_type', 'Unknown'),
                'replicate': params.get('replicate', 0),
                'seed': params.get('seed', 0)
            })
        
        if not data_points:
            print("No data available for plotting")
            return
        
        df = pd.DataFrame(data_points)
        
        plt.figure(figsize=figsize)
        
        # Define colors and markers
        styles = {
            'Human-Borzoi': {'color': 'green', 'marker': '^'},
            'Mouse-Borzoi': {'color': 'red', 'marker': 's'},
            'Human-Decima': {'color': 'purple', 'marker': 'D'},
            'Random-Init': {'color': 'blue', 'marker': 'o'}
        }
        
        for exp_type in df['experiment_type'].unique():
            type_df = df[df['experiment_type'] == exp_type]
            style = styles.get(exp_type, {'color': 'gray', 'marker': 'o'})
            
            plt.scatter(type_df['lr'], type_df['min_val_loss'],
                       c=style['color'], marker=style['marker'], s=100,
                       alpha=0.8, edgecolors='black', linewidth=0.5,
                       label=f"{exp_type} (n={len(type_df)})")
        
        plt.xscale('log')
        plt.xlabel('Learning Rate (log scale)')
        plt.ylabel('Minimum Validation Loss')
        plt.title('Minimum Validation Loss vs Learning Rate')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()
        return plt.gcf()
    
    def get_top_models_by_group(self, top_n=5):
        """Get top models by experiment type"""
        results = {}
        
        for exp_name, data in self.experiment_data.items():
            exp_type = data['params'].get('experiment_type', 'Unknown')
            
            if exp_type not in results:
                results[exp_type] = []
            
            # Get minimum validation loss
            if 'val_loss_min_value' in data['metrics']:
                min_loss = data['metrics']['val_loss_min_value']
                min_epoch = data['metrics']['val_loss_min_epoch']
            elif 'val_loss' in data['metrics'] and not data['metrics']['val_loss'].empty:
                val_loss_df = data['metrics']['val_loss']
                min_loss = val_loss_df['value'].min()
                min_idx = val_loss_df['value'].idxmin()
                min_epoch = val_loss_df.loc[min_idx, 'epoch']
            else:
                continue
            
            results[exp_type].append({
                'experiment': exp_name,
                'min_val_loss': min_loss,
                'min_epoch': min_epoch,
                'path': data['path'],
                'params': data['params']
            })
        
        # Sort each group
        for exp_type in results:
            results[exp_type].sort(key=lambda x: x['min_val_loss'])
        
        # Print results
        print(f"\n🏆 TOP {top_n} MODELS BY EXPERIMENT TYPE:")
        print("=" * 100)
        
        for exp_type, models in results.items():
            print(f"\n{exp_type.upper()}:")
            print("-" * 50)
            for i, model in enumerate(models[:top_n], 1):
                print(f"{i}. {model['experiment']}")
                print(f"   Min Val Loss: {model['min_val_loss']:.6f} (at epoch {model['min_epoch']:.1f})")
                print(f"   Replicate: {model['params'].get('replicate', 'N/A')}")
                print(f"   Seed: {model['params'].get('seed', 'N/A')}")
                print(f"   Path: {model['path']}")
                print()
        
        return results

# Usage
experiment_dirs = [
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250617_225114",
    "/hpc/scratch/group.data.science/mathias.voges/zebrahub-decima/experiments/decima_experiments_20250618_111138"
]

# Create analyzer and load experiments
analyzer = NewExperimentAnalyzer()
analyzer.load_experiment_dirs(experiment_dirs)

# Generate plots
analyzer.plot_training_curves(['val_loss', 'train_loss_epoch', 'val_pearson'])
plt.show()

analyzer.plot_min_val_loss_vs_lr()
plt.show()

# Get top models
top_models = analyzer.get_top_models_by_group(top_n=5)